# Per-scroll eval figures for the Campaign 31 baseline winner

Defaults to `31_early_gated_patch_groupdro_23_13-58-13` with its best-character-F1 checkpoint (`final_best_character.pth`).

Architecture, inference geometry, and scroll splits are loaded from the saved run config.

Layout per figure:

| inference | inference + inklabel overlay |
| 1.1um inklabel | 2.4um inklabel |

- the overlay uses the run's `inklabel_dir` (the labels it trained on)
- manual-split runs outline `train_masks/<id>.png` with a dotted contour; axis-split runs draw a dotted line
- inference uses one plain pass through the shared bounded-memory `predict_tiles` implementation
- one scroll is loaded, rendered, and released per cell
- run the setup cells first, then run scroll cells individually


In [6]:
import os
from pathlib import Path
EXP_NAME = os.getenv("VESUVIUS_EXP_NAME", "31_early_gated_patch_groupdro")
RUN_ROOT = Path(os.getenv("VESUVIUS_RUN_ROOT", "/vesuvius/runs_archs31"))
RUN_ID = os.getenv("VESUVIUS_RUN_ID", "31_early_gated_patch_groupdro_23_13-58-13")

In [7]:
# run selection: config and checkpoint both come from the same run
import json
import os
from pathlib import Path
RUN_DIR = RUN_ROOT / RUN_ID
RUN_CONFIG_PATH = RUN_DIR / "config.json"
if not RUN_CONFIG_PATH.is_file():
    candidates = sorted(RUN_ROOT.glob(f"{EXP_NAME}_*/config.json"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"no run config matching {EXP_NAME!r} under {RUN_ROOT}")
    RUN_CONFIG_PATH = candidates[-1]
    RUN_DIR = RUN_CONFIG_PATH.parent
    RUN_ID = RUN_DIR.name

with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as handle:
    RUN_CONFIG = json.load(handle)
_RUN_DATA = RUN_CONFIG["data"]
_RUN_MODEL = RUN_CONFIG["model"]

# a checkpoint from another run still loads strictly (convs are size-agnostic) but sees the wrong context geometry
MODEL_PATH = Path(os.getenv(
    "VESUVIUS_MODEL_PATH",
    str(Path("/vesuvius") / RUN_CONFIG["model_dir"] / "final_best_character.pth"),
))
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"checkpoint is missing: {MODEL_PATH}")
if Path(RUN_CONFIG["model_dir"]).name != MODEL_PATH.parent.name:
    raise RuntimeError(f"checkpoint {MODEL_PATH} does not belong to run model_dir {RUN_CONFIG['model_dir']}")

ARCH = _RUN_MODEL["arch"]
TILE_SIZE = int(_RUN_DATA["tile_size"])
CONTEXT_SIZE = int(_RUN_DATA["context_size"])
CONTEXT_DOWNSAMPLE = int(_RUN_DATA["context_downsample"])
DEPTH = int(_RUN_DATA["depth"])
D_START = int(_RUN_DATA["d_start"])
D_END = int(_RUN_DATA["d_end"])
MULTITILE = bool(_RUN_MODEL["multitile"])
MULTITILE_SUBTILE = int(_RUN_MODEL["multitile_subtile"])
MULTITILE_GRID = int(_RUN_MODEL["multitile_grid"])
TTA = False
TTA_MODE = _RUN_DATA["tta_mode"]
EVAL_CMAP_NORM = "raw"
EVAL_DPI = 200
OUTPUT_DIR = "output"
# inference window step in px (<= MULTITILE_GRID * MULTITILE_SUBTILE); 16 = densest, 32 = ~4x faster
EVAL_STRIDE = 32
_RUN_DATA["eval_stride"] = EVAL_STRIDE

EVAL_SCROLLS = {
    "w013": 20240304141531,
    "w018": 20240304144031,
    "500P2_front": 20250628074500,
    "seg46527": 20260226000000,
    "w035": 20260317000000,
    "w044": 20260115000000,
    "w059": 20250223000000,
    "w047": 20260206000001,
    "w056": 20260115000001,
    "w058": 20260210000000,
    "w052": 20260227000000,
    "w049": 20260318000000,
    "w046": 20260325000000,
    "w041": 20260108000000,
    "w040": 20250831000000,
    "w039": 20260302000000,
    "w038": 20260306000000,
    "w037": 20260310000000,
    "w034": 20260303000000,
    "w068": 20251111010954,
    "w087": 20251112000002,
    "p9b_487": 20250919125754,
    "paris4": 20231210121321,
    "paris2_fr143": 20230301213755,
    "paris2_fr47": 20230205142449,
    "scroll6_fr8": 20231205222200,
    "paris1_fr34": 20230301213423,
    "p343": 20250511003658,
    "cr1fr3": 20231201215900,
    "p841": 20260221022814,
}

if DEPTH != 8 or not _RUN_MODEL.get("surface_teacher_input") or not _RUN_DATA.get("surface_relative_depth_window"):
    raise RuntimeError("selected run is not literal-surface true slice-8")
if not _RUN_MODEL.get("early_2d_unet") or not _RUN_MODEL.get("gated_stems"):
    raise RuntimeError("selected run is not the early-2D gated baseline")
print(RUN_ID, MODEL_PATH, f"depth={DEPTH}", "literal surface", f"ctx={CONTEXT_SIZE}/ds{CONTEXT_DOWNSAMPLE}", f"stride={EVAL_STRIDE}")


31_early_gated_patch_groupdro_23_13-58-13 /vesuvius/models/archs31/early_gated_patch_groupdro/final_best_character.pth depth=8 literal surface ctx=192/ds2 stride=32


In [8]:
import gc
import importlib
import os
import sys
import types

REPO = "/vesuvius"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr

from utils.config import Config
from utils.dataloader import _load_unified_cache
from utils.model import create_model
from utils import visualizer as visualizer_module

# reload so an existing kernel cannot retain a stale inference reader
visualizer_module = importlib.reload(visualizer_module)
TBV = visualizer_module.TensorboardVisualizer
predict_tiles = visualizer_module.predict_tiles
group_by_depth = visualizer_module.group_by_depth
imread_gray = visualizer_module.imread_gray

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bounded-memory visualizer reloaded")

torch 2.11.0+cu128 | cuda True | bounded-memory visualizer reloaded


In [9]:
# config, strict model loading, split helpers, and per-scroll rendering
from utils.config import ScrollConfig
from utils.dataloader import DataManager

_AUXILIARY_PREFIXES = ("supcon_head.", "domain_head.")


def build_config():
    c = Config()
    # replay the full saved run config so geometry, surface window, and splits match training
    for name, value in _RUN_DATA.items():
        if name == "scrolls":
            c.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(c.data, name):
            setattr(c.data, name, value)
    c.data.eval_cmap_norm = EVAL_CMAP_NORM
    for name, value in _RUN_MODEL.items():
        if hasattr(c.model, name):
            setattr(c.model, name, value)
    c.model.compile_model = False
    c.tra.supcon = False
    c.tra.dann = False
    c.device = "cuda" if torch.cuda.is_available() else "cpu"
    return c


def _checkpoint_state(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    cleaned = {}
    for key, value in state.items():
        key = key.removeprefix("module.").removeprefix("_orig_mod.")
        if not key.startswith(_AUXILIARY_PREFIXES):
            cleaned[key] = value
    return cleaned, len(state) - len(cleaned)


def load_model(c):
    model, n = create_model(c)
    state, ignored = _checkpoint_state(MODEL_PATH)
    model.load_state_dict(state, strict=True)
    model.eval()
    print(f"loaded {MODEL_PATH} params={n:,} ignored_training_only={ignored}")
    return model


def _scroll_cfg(sid, c):
    return next((scroll for scroll in c.data.scrolls if int(scroll.scroll_id) == int(sid)), None)


def _splits(sid, height, width, c):
    tile = c.data.tile_size
    scroll = _scroll_cfg(sid, c)
    crop_x = scroll.crop_x_frac if scroll else (0.0, 1.0)
    crop_y = scroll.crop_y_frac if scroll else (0.0, 1.0)
    axis = (scroll.split_axis if scroll else "x").lower()
    fraction = scroll.train_split_frac if scroll else 0.75
    x0 = (int(width * crop_x[0]) // tile) * tile
    x1 = max((int(width * crop_x[1]) // tile) * tile, x0 + tile)
    y0 = (int(height * crop_y[0]) // tile) * tile
    y1 = max((int(height * crop_y[1]) // tile) * tile, y0 + tile)
    if axis == "y":
        split = (int((y1 - y0) * fraction) // tile) * tile
        train_range = (y0, y0 + split)
        full_y, full_x = (y0, y1), (x0, x1)
    else:
        split = (int((x1 - x0) * fraction) // tile) * tile
        train_range = (x0, x0 + split)
        full_y, full_x = (y0, y1), (x0, x1)
    return full_y, full_x, (train_range[1] - train_range[0]) // tile, axis


def _manual_train_grid(sid, full_y, full_x, grid_shape, c):
    """train cells from train_masks/<sid>.png, matching the training-time eval figure"""
    if bool(getattr(c.data, "simple_split", True)):
        return None
    train_mask = imread_gray(os.path.join(c.data.train_mask_dir, f"{sid}.png"))
    if train_mask is None:
        return None
    unit = int(c.model.multitile_subtile)
    aligned = DataManager._align_manual_mask(train_mask, unit)
    height, width = grid_shape
    region = aligned[full_y[0]:full_y[0] + height * unit, full_x[0]:full_x[0] + width * unit]
    region = np.pad(region, ((0, height * unit - region.shape[0]), (0, width * unit - region.shape[1])))
    return region.reshape(height, unit, width, unit).all(axis=(1, 3))


def get_norm(sid, vol, mask):
    stats = _load_unified_cache().get(str(sid))
    if stats and all(key in stats for key in ("mean", "std", "min", "max")):
        return stats["mean"], stats["std"], stats["min"], stats["max"]
    raise RuntimeError(f"normalization cache missing for {sid}")


def _run_name():
    return RUN_ID


def render_eval(name, sid, model, c):
    sid = int(sid)
    fake = types.SimpleNamespace(c=c)
    fake._display_norm = types.MethodType(TBV._display_norm, fake)
    vol = zarr.open(os.path.join(c.data.zarr_path, f"{sid}.zarr"), mode="r")
    height, width = int(vol.shape[1]), int(vol.shape[2])
    mask = (imread_gray(f"masks/{sid}.png") > 127).astype(np.uint8)
    # overlay the label set the run actually trained on
    label_dir = getattr(c.data, "inklabel_dir", None) or "./eroded_inklabels"
    labels = (imread_gray(os.path.join(label_dir, f"{sid}.png")) > 127).astype(np.uint8)
    common_height = min(height, mask.shape[0], labels.shape[0])
    common_width = min(width, mask.shape[1], labels.shape[1])
    vol = vol[:, :common_height, :common_width]
    mask = mask[:common_height, :common_width]
    labels = labels[:common_height, :common_width]
    mean, std, global_min, global_max = get_norm(sid, vol, mask)
    surface_dir = os.path.join(c.data.surface_label_dir, str(sid))
    surface_depth = np.load(os.path.join(surface_dir, "depth.npy"), mmap_mode="r")
    surface_confidence = np.load(os.path.join(surface_dir, "confidence.npy"), mmap_mode="r")

    full_y, full_x, train_split_n, split_axis = _splits(sid, common_height, common_width, c)
    coords = TBV._gen_tile_coords(
        fake, (c.data.d_start, c.data.d_end), full_y, full_x, mask, z_step=c.data.depth
    )
    grouped = group_by_depth(coords)
    if len(grouped) != 1:
        raise RuntimeError(f"surface-relative inference expected one depth pass, got {len(grouped)}")
    depth_offset = next(iter(grouped))
    result = predict_tiles(
        c, model, vol, mask, grouped[depth_offset], full_y, full_x,
        c.data.d_start + depth_offset, f"eval_{sid}", mean, std, global_min, global_max,
        also_tta=TTA, surface_depth_map=surface_depth, surface_confidence_map=surface_confidence,
    )
    reg_pred, tta_pred = result if TTA else (result, None)
    label_binary, _, _ = TBV._compute_tile_maps(fake, labels, mask, full_y, full_x)
    pred_height, pred_width = reg_pred.shape
    manual_split = not bool(getattr(c.data, "simple_split", True))
    manual_train_grid = _manual_train_grid(sid, full_y, full_x, reg_pred.shape, c)
    if manual_split:
        split_kind = "manual" if manual_train_grid is not None else "manual (no train mask)"
        # the axis split is not the training split here, so never draw its line
        train_split_n = split_axis = None
    else:
        split_kind = f"{split_axis}-split"

    def _raw(subdir):
        image = imread_gray(f"./inklabels/{subdir}/{sid}.png")
        if image is None:
            return None
        mh, mw = mask.shape
        rh, rw = image.shape
        crop = image[
            int(full_y[0] * rh / mh):int(full_y[1] * rh / mh),
            int(full_x[0] * rw / mw):int(full_x[1] * rw / mw),
        ]
        return cv2.resize(crop, (pred_width, pred_height), interpolation=cv2.INTER_LINEAR)

    fig = TBV._create_eval_figure_2x2(
        fake, reg_pred, tta_pred, label_binary, _raw("1_1um"), _raw("2_4um"),
        train_split_n, split_axis, manual_train_grid,
    )
    fig.suptitle(f"{name} (scroll {sid}) {split_kind} | labels {os.path.basename(os.path.normpath(label_dir))}", fontsize=10)
    out_dir = os.path.join(OUTPUT_DIR, "eval_visualizations", _run_name())
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}_{sid}.png")
    fig.savefig(out_path, dpi=EVAL_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {out_path} pred grid {reg_pred.shape}")
    del vol, mask, labels, reg_pred, tta_pred, label_binary, fig
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [10]:
# build config and load the model once
from utils.platform import get_zarr_dir

C = build_config()
C.data.zarr_path = get_zarr_dir()
MODEL = load_model(C)
print(
    "arch:", C.model.arch,
    "| zarr:", C.data.zarr_path,
    "| context:", C.data.context_size,
    "ds", C.data.context_downsample,
    "| feature_attn_mil:", C.model.feature_attn_mil,
    "| learned_surface:", C.model.learned_surface,
    "| new_surface:", C.model.new_learned_surface,
    "| multitile:", f"{C.model.multitile_subtile}x{C.model.multitile_grid}",
)

Model parameters (nnunet3d_lcndz): 7,522,714
loaded /vesuvius/models/archs31/early_gated_patch_groupdro/final_best_character.pth params=7,522,714 ignored_training_only=0
arch: nnunet3d_lcndz | zarr: /vesuvius/ves_zarrs2 | context: 192 ds 2 | feature_attn_mil: False | learned_surface: False | new_surface: False | multitile: 16x4


In [6]:
# 500P2_front — PHerc0500P2, split x 0.60
render_eval("500P2_front", EVAL_SCROLLS["500P2_front"], MODEL, C)


[predict] reading 12471 tiles in 167 row-strips (eval_20250628074500)


[predict] eval_20250628074500 done in   9.4s  predict   8.3s (88.2%)  read/wait   1.1s (11.8%)  [prefetch x3]
[saved] output/eval_visualizations/35_holdout_dual_scale_25_11-37-12/500P2_front_20250628074500.png pred grid (392, 223)


In [7]:
# w035 — split y 0.75
render_eval("w035", EVAL_SCROLLS["w035"], MODEL, C)

[predict] reading 24552 tiles in 178 row-strips (eval_20260317000000)


[predict] eval_20260317000000 done in  16.6s  predict  14.8s (89.1%)  read/wait   1.8s (10.9%)  [prefetch x3]
[saved] output/eval_visualizations/35_holdout_dual_scale_25_11-37-12/w035_20260317000000.png pred grid (363, 327)


In [ ]:
# w044 — split y 0.8055
render_eval("w044", EVAL_SCROLLS["w044"], MODEL, C)


In [ ]:
# w059 — split x 0.75
render_eval("w059", EVAL_SCROLLS["w059"], MODEL, C)


In [ ]:
# w047 — split x 0.75
render_eval("w047", EVAL_SCROLLS["w047"], MODEL, C)


In [ ]:
# w056 — split y 0.50
render_eval("w056", EVAL_SCROLLS["w056"], MODEL, C)


In [ ]:
# w058 — split x 0.75
render_eval("w058", EVAL_SCROLLS["w058"], MODEL, C)


In [ ]:
# w052 — split x 0.75
render_eval("w052", EVAL_SCROLLS["w052"], MODEL, C)


In [ ]:
# w049 — split x 0.75
render_eval("w049", EVAL_SCROLLS["w049"], MODEL, C)


In [ ]:
# w046 — split x 0.75
render_eval("w046", EVAL_SCROLLS["w046"], MODEL, C)


In [ ]:
# w041 — split x 0.75
render_eval("w041", EVAL_SCROLLS["w041"], MODEL, C)


In [ ]:
# w040 — split x 0.75
render_eval("w040", EVAL_SCROLLS["w040"], MODEL, C)


In [ ]:
# w039 — split x 0.75
render_eval("w039", EVAL_SCROLLS["w039"], MODEL, C)


In [ ]:
# w038 — split x 0.75
render_eval("w038", EVAL_SCROLLS["w038"], MODEL, C)


In [ ]:
# w037 — split x 0.75
render_eval("w037", EVAL_SCROLLS["w037"], MODEL, C)


In [ ]:
# w034 — split x 0.75
render_eval("w034", EVAL_SCROLLS["w034"], MODEL, C)


In [11]:
# seg46527 — PHerc0814
render_eval("seg46527", EVAL_SCROLLS["seg46527"], MODEL, C)

[predict] reading 4368 tiles in 66 row-strips (eval_20260226000000)


[predict] eval_20260226000000 done in   4.3s  predict   2.7s (61.6%)  read/wait   1.7s (38.4%)
[saved] output/eval_visualizations/31_early_gated_patch_groupdro_23_13-58-13/seg46527_20260226000000.png pred grid (136, 222)


In [ ]:
# PHercParis2 Fr143
render_eval("paris2_fr143", EVAL_SCROLLS["paris2_fr143"], MODEL, C)

In [ ]:
# PHercParis2 Fr47
render_eval("paris2_fr47", EVAL_SCROLLS["paris2_fr47"], MODEL, C)


In [ ]:
# PHerc51 Cr4 Fr8
render_eval("scroll6_fr8", EVAL_SCROLLS["scroll6_fr8"], MODEL, C)

In [ ]:
# PHercParis1 Fr34
render_eval("paris1_fr34", EVAL_SCROLLS["paris1_fr34"], MODEL, C)

In [ ]:
# PHerc0343P
render_eval("p343", EVAL_SCROLLS["p343"], MODEL, C)

In [ ]:
# PHerc0841
render_eval("p841", EVAL_SCROLLS["p841"], MODEL, C)

In [ ]:
# Pherc1667 w018 (large)
render_eval("w018", EVAL_SCROLLS["w018"], MODEL, C)

In [ ]:
# Pherc1667 w013 (small)
render_eval("w013", EVAL_SCROLLS["w013"], MODEL, C)

In [ ]:
# PHerc1667 Cr1 Fr3 (frag)
render_eval("cr1fr3", EVAL_SCROLLS["cr1fr3"], MODEL, C)

In [9]:
# PhercParis4
render_eval("paris4", EVAL_SCROLLS["paris4"], MODEL, C)

[predict] reading 107735 tiles in 397 row-strips (eval_20231210121321)


Read eval_20231210121321:  18%|█▊        | 73/397 [00:51<02:17,  2.35it/s]

KeyboardInterrupt: 

In [7]:
# Pherc0009b
render_eval("p9b_487", EVAL_SCROLLS["p9b_487"], MODEL, C)

[predict] reading 25650 tiles in 193 row-strips (eval_20250919125754)


KeyboardInterrupt: 

In [ ]:
# Pherc0172 w068 (faint)
render_eval("w068", EVAL_SCROLLS["w068"], MODEL, C)

In [ ]:
# Pherc0172 w087 (solid)
render_eval("w087", EVAL_SCROLLS["w087"], MODEL, C)